In [ ]:
from google.colab import userdata
import os

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('YC_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('YC_SECRET_ACCESS_KEY')
S3_BUCKET   = userdata.get('S3_BUCKET')
S3_ENDPOINT = 'https://storage.yandexcloud.net'

In [ ]:
os.system('pip install awscli -q')
os.system(f'aws s3 cp s3://{S3_BUCKET}/second/cv/dataset_processed.tar.gz /content/dataset_processed.tar.gz --endpoint-url {S3_ENDPOINT}')
os.system('tar -xzf /content/dataset_processed.tar.gz -C /content/')
print('dataset ready')

In [ ]:
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

In [ ]:
PROCESSED_DIR = Path('/content/dataset_processed')

RUN         = 'efficientnet_b0_2'
WEIGHTS_DIR = Path(f'/content/weights/{RUN}')
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

CLASSES     = ['gun', 'spg', 'ifv', 'uav', 'armored_vehicle', 'apc', 'infantry', 'mlrs', 'tank']
NUM_CLASSES = len(CLASSES)
BATCH_SIZE  = 32
EPOCHS      = 30
LR          = 3e-4
WEIGHT_DECAY= 1e-4
TRAIN_RATIO = 0.7
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f'device: {DEVICE}  run: {RUN}')

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

only_root_jpg = lambda p: Path(p).parent.parent == PROCESSED_DIR and p.endswith('.jpg')

full_train = datasets.ImageFolder(PROCESSED_DIR, transform=train_tf, is_valid_file=only_root_jpg)
full_test  = datasets.ImageFolder(PROCESSED_DIR, transform=test_tf,  is_valid_file=only_root_jpg)

n_train  = int(len(full_train) * TRAIN_RATIO)
n_test   = len(full_train) - n_train
indices  = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(67)).tolist()

train_set = torch.utils.data.Subset(full_train, indices[:n_train])
test_set  = torch.utils.data.Subset(full_test,  indices[n_train:])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

counts  = torch.zeros(NUM_CLASSES)
for _, label in train_set:
    counts[label] += 1
class_weights = (counts.sum() / (NUM_CLASSES * counts)).to(DEVICE)

print(f'train: {n_train}  test: {n_test}')
print('class weights:', {full_train.classes[i]: f'{class_weights[i]:.2f}' for i in range(NUM_CLASSES)})

In [ ]:
def build_simple_cnn() -> nn.Module:
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(128 * 28 * 28, 256), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(256, NUM_CLASSES),
    )

def build_resnet50() -> nn.Module:
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def build_efficientnet_b0() -> nn.Module:
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    return m

def build_efficientnet_b2() -> nn.Module:
    m = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    return m

In [ ]:
def train_model(name: str, model: nn.Module) -> None:
    print(f"{'='*40}\n  {name}\n{'='*40}")

    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        t0 = time.time()

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += images.size(0)

        scheduler.step()
        print(f"  epoch {epoch:>2}/{EPOCHS}  loss {total_loss/total:.4f}  acc {correct/total*100:.1f}%  lr {scheduler.get_last_lr()[0]:.2e}  ({time.time()-t0:.0f}s)")

    local_path = WEIGHTS_DIR / f'{name}.pt'
    torch.save(model.state_dict(), local_path)
    os.system(f'aws s3 cp {local_path} s3://{S3_BUCKET}/second/cv/weights/{RUN}/{name}.pt --endpoint-url {S3_ENDPOINT}')
    print(f"  {name}.pt -> s3/{RUN}\n")

In [ ]:
train_model('efficientnet_b0', build_efficientnet_b0())